<a href="https://colab.research.google.com/github/leonidasf300/OUU2026/blob/main/LeonSerna_OUU_project_v0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installing libraries

In [1]:
pip install pandapower[all]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.5/316.5 kB 12.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.9/292.9 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 25.7 MB/s eta 0:00:00


# Importing libraries

In [2]:
import pandapower as pp
import pandapower.shortcircuit as sc
import pandapower as pp
import pandapower.shortcircuit as sc
import pandas as pd


# Creating IEC MG

In [14]:


def create_iec_microgrid():
    # Crear red vacía
    net = pp.create_empty_network(f_hz=60.0)

    # ================= BUSES =================
    # Red de utilidad
    bus_utility = pp.create_bus(net, vn_kv=120.0, name="Utility 120kV")

    # Nivel de distribución 25 kV
    bus_pcc = pp.create_bus(net, vn_kv=25.0, name="PCC B1")
    bus2 = pp.create_bus(net, vn_kv=25.0, name="BUS 2")
    bus3 = pp.create_bus(net, vn_kv=25.0, name="BUS 3")
    bus4 = pp.create_bus(net, vn_kv=25.0, name="BUS 4")
    bus5 = pp.create_bus(net, vn_kv=25.0, name="BUS 5")
    bus6 = pp.create_bus(net, vn_kv=25.0, name="BUS 6")

    # Buses de Generadores (Baja Tensión)
    bus_dg1 = pp.create_bus(net, vn_kv=2.4, name="Terminal DG1")
    bus_dg2 = pp.create_bus(net, vn_kv=2.4, name="Terminal DG2")
    bus_dg3 = pp.create_bus(net, vn_kv=0.575, name="Terminal DG3")
    bus_dg4 = pp.create_bus(net, vn_kv=0.575, name="Terminal DG4")

    # ================= GRID EXTERNO =================
    # S_sc = 1000 MVA según Tabla 1
    pp.create_ext_grid(net, bus_utility, s_sc_max_mva=1000.0, rx_max=0.1, name="Utility Grid")

    # ================= TRANSFORMADORES =================
    # vkr_percent = 0.00375 * 100 = 0.375%
    # vk_percent = sqrt(0.00375^2 + 0.01^2) * 100 = 1.068%
    # Calc: vkr=R1*100 | vk=sqrt(R1**2+X1**2)*100 | pfe=(1/Rm)*Sn*1e3 | i0=sqrt((1/Rm)**2+(1/Xm)**2)*100

    # TR-1: 15 MVA, 120/25 kV
    pp.create_transformer_from_parameters(net, hv_bus=bus_utility, lv_bus=bus_pcc, sn_mva=15.0,
                                          vn_hv_kv=120.0, vn_lv_kv=25.0, vkr_percent=0.375,
                                          vk_percent=1.068, pfe_kw=30, i0_percent=0.2828, name="TR-1")

    # TR-2 y TR-3: 12 MVA, 25/2.4 kV
    pp.create_transformer_from_parameters(net, hv_bus=bus2, lv_bus=bus_dg1, sn_mva=12.0,
                                          vn_hv_kv=25.0, vn_lv_kv=2.4, vkr_percent=0.375,
                                          vk_percent=1.068, pfe_kw=24, i0_percent=0.2828, name="TR-2")
    pp.create_transformer_from_parameters(net, hv_bus=bus3, lv_bus=bus_dg2, sn_mva=12.0,
                                          vn_hv_kv=25.0, vn_lv_kv=2.4, vkr_percent=0.375,
                                          vk_percent=1.068, pfe_kw=24, i0_percent=0.2828, name="TR-3")

    # TR-4 y TR-5: 10 MVA, 25/0.575 kV
    pp.create_transformer_from_parameters(net, hv_bus=bus4, lv_bus=bus_dg3, sn_mva=10.0,
                                          vn_hv_kv=25.0, vn_lv_kv=0.575, vkr_percent=0.375,
                                          vk_percent=1.068, pfe_kw=20, i0_percent=0.2828, name="TR-4")
    pp.create_transformer_from_parameters(net, hv_bus=bus6, lv_bus=bus_dg4, sn_mva=10.0,
                                          vn_hv_kv=25.0, vn_lv_kv=0.575, vkr_percent=0.375,
                                          vk_percent=1.068, pfe_kw=20, i0_percent=0.2828, name="TR-5")

    # ================= LÍNEAS DE DISTRIBUCIÓN =================
    # R1 = 0.413 ohm/km, L1 = 3.32e-3 H/km -> X1 = 2*pi*60*3.32e-3 = 1.251 ohm/km
    # C1 = 5.01e-9 F/km
    line_params = {
    "r_ohm_per_km": 0.413,
    "x_ohm_per_km": 1.251,
    "c_nf_per_km": 5.01,
    "r0_ohm_per_km": 0.1153, # Datos de la Tabla 5
    "x0_ohm_per_km": 0.396,  # Calculado de L0
    "c0_nf_per_km": 11.33,   # Datos de la Tabla 5
    "max_i_ka": 1.0
}

    pp.create_line_from_parameters(net, from_bus=bus_pcc, to_bus=bus2, length_km=30.0, name="DL-1", **line_params)
    pp.create_line_from_parameters(net, from_bus=bus_pcc, to_bus=bus3, length_km=30.0, name="DL-2", **line_params)
    pp.create_line_from_parameters(net, from_bus=bus3, to_bus=bus4, length_km=30.0, name="DL-3", **line_params)
    pp.create_line_from_parameters(net, from_bus=bus3, to_bus=bus5, length_km=30.0, name="DL-4", **line_params)
    pp.create_line_from_parameters(net, from_bus=bus5, to_bus=bus6, length_km=30.0, name="DL-5", **line_params)

    # Interruptores de lazo (Loop breakers) - Normalmente abiertos para operación radial
    #pp.create_switch(net, bus=bus2, element=bus4, et="b", closed=False, name="CB_LOOP 1")
    #pp.create_switch(net, bus=bus4, element=bus6, et="b", closed=False, name="CB_LOOP 2")

    # ================= GENERADORES DISTRIBUIDOS =================
    # DG1, DG2: Sincrónicos (9 MVA, xd'' = 0.177)
    pp.create_gen(net, bus_dg1, p_mw=8.1, vm_pu=1.0, sn_mva=9.0, vn_kv=2.4, xdss_pu=0.177, rdss_ohm=0.0, cos_phi=0.9, name="DG1")
    pp.create_gen(net, bus_dg2, p_mw=8.1, vm_pu=1.0, sn_mva=9.0, vn_kv=2.4, xdss_pu=0.177, rdss_ohm=0.0, cos_phi=0.9, name="DG2")

    # DG3: Inversor Eólico (6 MVA, xd'' = 0.252)
    pp.create_gen(net, bus_dg3, p_mw=5.4, vm_pu=1.0, sn_mva=6.0, vn_kv=0.575, xdss_pu=0.252, rdss_ohm=0.0, cos_phi=0.9, name="DG3")

    # DG4: DFIG Eólico (9 MVA, Ls + Lr' = 0.34)
    pp.create_gen(net, bus_dg4, p_mw=8.1, vm_pu=1.0, sn_mva=9.0, vn_kv=0.575, xdss_pu=0.34, rdss_ohm=0.0, cos_phi=0.9, name="DG4")

    # ================= CARGAS =================
    # Total de 22 MW y 10 MVAR dividido en 6 cargas
    p_load = 22.0 / 6.0
    q_load = 10.0 / 6.0

    pp.create_load(net, bus_pcc, p_mw=p_load, q_mvar=q_load, name="L-1")
    pp.create_load(net, bus2, p_mw=p_load, q_mvar=q_load, name="L-2")
    pp.create_load(net, bus3, p_mw=p_load, q_mvar=q_load, name="L-3")
    pp.create_load(net, bus4, p_mw=p_load, q_mvar=q_load, name="L-4")
    pp.create_load(net, bus5, p_mw=p_load, q_mvar=q_load, name="L-5")
    pp.create_load(net, bus6, p_mw=p_load, q_mvar=q_load, name="L-6")

    return net

# Inicializar red
net = create_iec_microgrid()

# ================= ANÁLISIS DE CORTOCIRCUITO =================
# Ejecutar cortocircuito trifásico según norma IEC 60909
# case="max" para calcular corrientes máximas de falla
sc.calc_sc(net, case="max", ip=True, ith=True, branch_results=True)

# Imprimir resultados en buses (corriente de cortocircuito subtransitoria ikss)
print("--- Resultados de Cortocircuito en Buses (ikss en kA) ---")
print(net.res_bus_sc[['ikss_ka', 'ip_ka']])

--- Resultados de Cortocircuito en Buses (ikss en kA) ---
      ikss_ka      ip_ka
0    4.938037  12.162243
1   14.547812  31.822484
2    1.589956   3.779257
3    1.986356   4.655884
4    0.915415   2.109035
5    0.585872   1.253501
6    0.857791   2.038392
7   17.035906  40.783085
8   20.985762  49.446725
9   40.221732  92.932948
10  38.066817  90.829982


/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)


In [ ]:
#ICC = ICC4.copy()

BACKUP = [[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0],
          [0,0,0,1,0,0,0,0,0,0,0,0,0,0,0],
          [1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
          [0,0,0,0,0,1,0,0,0,0,0,0,0,0,1],
          [0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
          [0,0,0,0,0,0,1,1,0,0,0,0,0,0,0],
          [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
          [0,0,0,0,0,0,0,0,0,0,1,0,0,0,0],#M8, B11
          [0,0,0,0,0,0,0,0,0,0,0,0,0,1,0],
          [0,0,0,0,0,1,0,0,0,0,0,0,0,0,1],
          [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
          [0,0,0,0,1,0,1,0,0,0,0,0,0,0,0],
          [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
          [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
          [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]]

CT_PS = [[400,0.5],
          [400,0.5],
          [400,0.5],
          [400,0.5],
          [400,0.5],
          [400,0.5],
          [1200,1],
          [400,0.5],
          [400,0.5],
          [400,0.5],
          [400,0.65],
          [400,0.5],
          [400,0.88],
          [400,0.65],
          [400,0.55]]

principal = [[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0],
              [0,0,1,1,0,0,0,0,0,0,0,0,0,0,0],
              [0,0,0,0,1,1,0,0,0,0,0,0,0,0,0],
              [0,0,0,0,0,0,0,1,0,0,0,1,0,0,0],
              [0,0,0,0,0,0,0,0,1,1,0,0,0,0,0],]



In [1]:
headers = ["DG0", "DG1", "DG2", "DG3","DG4"]
rows = ["OM1","OM2","OM3","OM4"]

matrix = [[1,0,0,0,0],
          [1,1,1,1,1],
          [1,1,1,0,0],
          [0,1,1,1,1]]

OMS = {
    (rows[i], headers[j]): matrix[i][j]
    for i in range(len(rows))
    for j in range(len(headers))
}
OMS


{('OM1', 'DG0'): 1,
 ('OM1', 'DG1'): 0,
 ('OM1', 'DG2'): 0,
 ('OM1', 'DG3'): 0,
 ('OM1', 'DG4'): 0,
 ('OM2', 'DG0'): 1,
 ('OM2', 'DG1'): 1,
 ('OM2', 'DG2'): 1,
 ('OM2', 'DG3'): 1,
 ('OM2', 'DG4'): 1,
 ('OM3', 'DG0'): 1,
 ('OM3', 'DG1'): 1,
 ('OM3', 'DG2'): 1,
 ('OM3', 'DG3'): 0,
 ('OM3', 'DG4'): 0,
 ('OM4', 'DG0'): 0,
 ('OM4', 'DG1'): 1,
 ('OM4', 'DG2'): 1,
 ('OM4', 'DG3'): 1,
 ('OM4', 'DG4'): 1}

In [15]:
net.res_bus_sc

,ikss_ka,skss_mw,ip_ka,ith_ka,rk_ohm,xk_ohm
0,4.938037,1026.351778,12.162243,5.019943,1.598697,15.350279
1,14.547812,629.938718,31.822484,14.667794,0.223209,1.068307
2,1.589956,68.847107,3.779257,1.610497,0.880130,9.947033
3,1.986356,86.011748,4.655884,2.009895,0.907701,7.941387
4,0.915415,39.638621,2.109035,0.925238,1.810208,17.249472
5,0.585872,25.369010,1.253501,0.590243,6.633126,26.275678
6,0.857791,37.143445,2.038392,0.868854,1.301650,18.463497
7,17.035906,70.816933,40.783085,17.266414,0.006558,0.089229
8,20.985762,87.236172,49.446725,21.242425,0.007317,0.072261
9,40.221732,40.057998,92.932948,40.660319,0.000875,0.009037


In [16]:
net.bus

,name,vn_kv,type,zone,in_service,geo
0,Utility 120kV,120.000,b,None,True,None
1,PCC B1,25.000,b,None,True,None
2,BUS 2,25.000,b,None,True,None
3,BUS 3,25.000,b,None,True,None
4,BUS 4,25.000,b,None,True,None
5,BUS 5,25.000,b,None,True,None
6,BUS 6,25.000,b,None,True,None
7,Terminal DG1,2.400,b,None,True,None
8,Terminal DG2,2.400,b,None,True,None
9,Terminal DG3,0.575,b,None,True,None


In [17]:
net.res_line_sc

,ikss_ka,ikss_from_ka,ikss_from_degree,ikss_to_ka,ikss_to_degree,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,ip_ka,ith_ka
0,0.390774,0.390774,-71.908436,0.390774,108.091564,5.676012,17.192956,3.443171,10.429557,1.097170,-0.046853,1.099359,-0.019814,0.928852,0.395822
1,0.390761,0.390761,-71.909607,0.390761,108.090393,5.675637,17.191819,3.877534,11.745267,1.097170,-0.046853,1.094309,-0.096968,0.915917,0.395391
2,0.327698,0.327698,-74.039961,0.327698,105.960039,3.991534,12.090578,2.188838,6.630113,1.094309,-0.096968,1.097859,-0.061017,0.754987,0.331214
3,0.330600,0.330600,-73.953379,0.330600,106.046621,4.062553,12.305699,0.928237,2.796306,1.094309,-0.096968,1.096570,-0.067231,0.707335,0.333067
4,0.255679,0.255679,101.738460,0.255679,-78.261540,1.227501,3.709595,2.429876,7.360230,1.096570,-0.067231,1.098832,-0.037618,0.547038,0.257587


In [18]:
sc.calc_sc(net, bus=2, fault='3ph', case='max',
           branch_results=True, r_fault_ohm=0.0, x_fault_ohm=0.0)

/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)


In [19]:
net.res_line_sc

,ikss_ka,ikss_from_ka,ikss_from_degree,ikss_to_ka,ikss_to_degree,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree
0,0.390774,0.390774,-71.908436,0.390774,108.091564,5.676012,17.192956,0.000000,-0.000000,1.070011,-0.178347,0.000000,0.000000
1,0.008861,0.008861,111.709798,0.008861,-68.290202,-0.153054,-0.380962,0.155973,0.389802,1.070011,-0.178347,1.094227,-0.098170
2,0.001339,0.001339,119.120437,0.001339,-60.879563,-0.030978,-0.055386,0.031045,0.055588,1.094227,-0.098170,1.097828,-0.061817
3,0.000864,0.000864,122.265179,0.000864,-57.734821,-0.021905,-0.034566,0.021933,0.034650,1.094227,-0.098170,1.096521,-0.068080
4,0.000864,0.000864,122.265179,0.000864,-57.734821,-0.021933,-0.034650,0.021961,0.034734,1.096521,-0.068080,1.098815,-0.038116


# Calculate SC porcentual

In [20]:


def calcular_sc_porcentual(net, nombre_linea, pasos=10):
    # 1. Identificar la línea original
    line_idx = net.line[net.line.name == nombre_linea].index[0]
    bus_inicial = net.line.at[line_idx, "from_bus"]
    bus_final = net.line.at[line_idx, "to_bus"]
    L_total = net.line.at[line_idx, "length_km"]
    params = {
        "r_ohm_per_km": net.line.at[line_idx, "r_ohm_per_km"],
        "x_ohm_per_km": net.line.at[line_idx, "x_ohm_per_km"],
        "c_nf_per_km": net.line.at[line_idx, "c_nf_per_km"],
        "max_i_ka": net.line.at[line_idx, "max_i_ka"]
    }

    # 2. Desactivar la línea original
    net.line.at[line_idx, "in_service"] = False

    resultados = []

    # 3. Iterar cada 10% (o el paso definido)
    for i in range(1, pasos):
        porcentaje = i / pasos
        distancia_falla = L_total * porcentaje

        # Crear bus temporal de falla
        f_bus = pp.create_bus(net, vn_kv=net.bus.at[bus_inicial, "vn_kv"], name=f"Falla_{int(porcentaje*100)}%")

        # Crear los dos tramos de línea que conectan al punto de falla
        l1 = pp.create_line_from_parameters(net, bus_inicial, f_bus, distancia_falla, name="tramo_A", **params)
        l2 = pp.create_line_from_parameters(net, f_bus, bus_final, L_total - distancia_falla, name="tramo_B", **params)

        # 4. Ejecutar SC con branch_results=True para ver qué ven los relés en los extremos
        sc.calc_sc(net, bus=f_bus, branch_results=True)

        # Guardar datos: I_total en falla, I que ve Relé inicial (l1), I que ve Relé final (l2)
        resultados.append({
            "Porcentaje": f"{int(porcentaje*100)}%",
            "I_falla_total": net.res_bus_sc.ikss_ka.at[f_bus],
            "I_Relé_Front": net.res_line_sc.ikss_ka.at[l1],
            "I_Relé_Back": net.res_line_sc.ikss_ka.at[l2]
        })

        # Limpiar para no acumular líneas en el modelo
        pp.drop_lines(net, [l1, l2])
        pp.drop_buses(net, [f_bus])

    # Restaurar línea original
    net.line.at[line_idx, "in_service"] = True
    return pd.DataFrame(resultados)

# Uso:
df_falla_dl1 = calcular_sc_porcentual(net, "DL-1")
print(df_falla_dl1)

/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprec

  Porcentaje  I_falla_total  I_Relé_Front  I_Relé_Back
0        10%       3.465725      3.136972     0.329224
1        20%       2.119288      1.761614     0.358498
2        30%       1.616958      1.224635     0.393459
3        40%       1.372978      0.938538     0.435933
4        50%       1.247455      0.760799     0.488610
5        60%       1.192692      0.639660     0.555631
6        70%       1.191926      0.551799     0.643688
7        80%       1.244364      0.485160     0.764311
8        90%       1.364085      0.432882     0.939031


In [21]:
0.760799   +  0.488610

1.249409

In [22]:
df_falla_dl1 = calcular_sc_porcentual(net, "DL-1")
print(df_falla_dl1)

/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprec

  Porcentaje  I_falla_total  I_Relé_Front  I_Relé_Back
0        10%       3.465725      3.136972     0.329224
1        20%       2.119288      1.761614     0.358498
2        30%       1.616958      1.224635     0.393459
3        40%       1.372978      0.938538     0.435933
4        50%       1.247455      0.760799     0.488610
5        60%       1.192692      0.639660     0.555631
6        70%       1.191926      0.551799     0.643688
7        80%       1.244364      0.485160     0.764311
8        90%       1.364085      0.432882     0.939031


In [23]:
df_falla_dl1 = calcular_sc_porcentual(net, "DL-2")
print(df_falla_dl1)

/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprec

  Porcentaje  I_falla_total  I_Relé_Front  I_Relé_Back
0        10%       3.487168      3.136128     0.351180
1        20%       2.145773      1.761350     0.384759
2        30%       1.649421      1.224507     0.425423
3        40%       1.413426      0.938463     0.475671
4        50%       1.299111      0.760750     0.539330
5        60%       1.260846      0.639626     0.622568
6        70%       1.285816      0.551774     0.735984
7        80%       1.381594      0.485140     0.899423
8        90%       1.582496      0.432866     1.154622


In [24]:
df_falla_dl1 = calcular_sc_porcentual(net, "DL-3")
print(df_falla_dl1)

/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprec

  Porcentaje  I_falla_total  I_Relé_Front  I_Relé_Back
0        10%       1.480475      1.222380     0.258131
1        20%       1.214724      0.939116     0.275679
2        30%       1.057356      0.761960     0.295757
3        40%       0.959032      0.640857     0.318946
4        50%       0.897622      0.552893     0.346017
5        60%       0.862203      0.486124     0.378014
6        70%       0.847333      0.433723     0.416377
7        80%       0.850785      0.391509     0.463152
8        90%       0.872681      0.356775     0.521322


In [25]:
df_falla_dl1 = calcular_sc_porcentual(net, "DL-4")
print(df_falla_dl1)

/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprec

  Porcentaje  I_falla_total  I_Relé_Front  I_Relé_Back
0        10%       1.426700      1.264314     0.162807
1        20%       1.133165      0.963603     0.169664
2        30%       0.955074      0.777962     0.177123
3        40%       0.837380      0.652118     0.185265
4        50%       0.755400      0.561242     0.194188
5        60%       0.696491      0.492559     0.204011
6        70%       0.653568      0.438833     0.214876
7        80%       0.622405      0.395664     0.226956
8        90%       0.600385      0.360220     0.240468


In [26]:
df_falla_dl1 = calcular_sc_porcentual(net, "DL-5")
print(df_falla_dl1)

/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprec

  Porcentaje  I_falla_total  I_Relé_Front  I_Relé_Back
0        10%       0.577877      0.305478     0.272930
1        20%       0.575876      0.283903     0.292655
2        30%       0.579729      0.265173     0.315422
3        40%       0.589647      0.248760     0.341986
4        50%       0.606228      0.234260     0.373366
5        60%       0.630542      0.221356     0.410977
6        70%       0.664308      0.209800     0.456836
7        80%       0.710211      0.199390     0.513903
8        90%       0.772457      0.189964     0.586678


In [29]:
# --- MODO ISLA (Islanding) ---
# 1. Desactivar la red externa (asumiendo que solo tienes una)
net.ext_grid.in_service = False

# 2. Correr tu función de fallas porcentuales
df_falla_isla_dl1 = calcular_sc_porcentual(net, "DL-1")

# 3. Mostrar resultados y restaurar
print("Resultados en Modo Isla (Sin Utility):")
print(df_falla_isla_dl1)

net.ext_grid.in_service = True


/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
/usr/local/lib/python3.12/dist-packages/pandapower/build_branch.py:1434: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprec

Resultados en Modo Isla (Sin Utility):
  Porcentaje  I_falla_total  I_Relé_Front  I_Relé_Back
0        10%       0.628106      0.298975     0.329224
1        20%       0.636626      0.278286     0.358498
2        30%       0.653482      0.260272     0.393459
3        40%       0.680003      0.244447     0.435933
4        50%       0.718484      0.230435     0.488610
5        60%       0.772738      0.217941     0.555631
6        70%       0.849160      0.206732     0.643688
7        80%       0.958963      0.196619     0.764311
8        90%       1.123229      0.187449     0.939031
